# ⚙️ Phase 2 — Final Week: Cross-Validation + Hyperparameter Tuning + Sklearn Pipelines
### This is how production ML code actually looks

---

**What you'll learn:**
- Cross-validation — a smarter way to evaluate than a single train/test split
- Hyperparameter tuning — automatically finding the best model settings
- Sklearn Pipeline — wrapping preprocessing + model into one clean object
- How these three things connect into a real ML workflow

> 💡 If Phase 2 so far was learning to cook individual dishes,
> this week is learning to run a proper kitchen.
> Everything becomes cleaner, more reliable, and production-ready.

---
## 🧠 Section 1: The Problem With a Single Train/Test Split

Every notebook so far used this pattern:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

This works — but has a hidden problem.

**The split is just one random cut of your data.**
What if by luck, all the easy samples landed in the test set?
Your F1 score would look great — but it's not the real picture.

What if all the hard samples landed in the test set?
Your score would look terrible — also not the real picture.

**The fix: Cross-Validation**

Instead of one split, split the data into **K folds** (chunks).
Train and evaluate K times — each time using a different fold as the test set.
Average the K scores → much more reliable estimate of real performance.

```
K=5 folds:
Run 1: [TEST][train][train][train][train] → F1: 0.88
Run 2: [train][TEST][train][train][train] → F1: 0.91
Run 3: [train][train][TEST][train][train] → F1: 0.87
Run 4: [train][train][train][TEST][train] → F1: 0.90
Run 5: [train][train][train][train][TEST] → F1: 0.89
                                     Average → F1: 0.89  ✅ reliable!
```

**JS analogy:** instead of running your test suite once,
you run it 5 times with different data each time and average the results.
Much more confidence in the outcome.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import f1_score

np.random.seed(42)
n = 1500

df = pd.DataFrame({
    "likes":          np.random.randint(50, 20000, n),
    "comments":       np.random.randint(5, 2000, n),
    "shares":         np.random.randint(2, 1500, n),
    "hashtag_count":  np.random.randint(0, 30, n),
    "caption_length": np.random.randint(10, 300, n),
    "hour_posted":    np.random.randint(0, 24, n),
    "follower_count": np.random.randint(100, 500000, n),
    "is_video":       np.random.randint(0, 2, n),
})
df["engagement_rate"] = (df["likes"] + df["comments"]) / (df["follower_count"] + 1)
df["shares_per_like"] = df["shares"] / (df["likes"] + 1)
df["engagement"] = df["likes"] + df["comments"] * 2 + df["shares"] * 3
threshold = df["engagement"].quantile(0.80)
df["is_viral"] = (df["engagement"] >= threshold).astype(int)
df = df.drop("engagement", axis=1)

X = df.drop("is_viral", axis=1)
y = df["is_viral"]

print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Viral posts: {y.sum()} ({y.mean()*100:.1f}%)")


In [ ]:
# ✅ Cross-validation in sklearn — one function call

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# cv=5 means 5-fold cross validation
# scoring='f1' means evaluate using F1 score each fold
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='f1')

print("=== 5-FOLD CROSS VALIDATION RESULTS ===")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: {score*100:.2f}%")

print(f"\n  Mean F1:  {cv_scores.mean()*100:.2f}%")
print(f"  Std Dev:  {cv_scores.std()*100:.2f}%")
print()
print("💡 Std Dev tells you how stable the model is.")
print("   Low std = consistent across different data slices = trustworthy.")
print("   High std = performance varies a lot = might be overfitting.")


In [ ]:
# ✅ Compare single split vs cross-validation

# Single split (old way)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf.fit(X_train, y_train)
single_f1 = f1_score(y_test, rf.predict(X_test))

print("=== Single Split vs Cross-Validation ===")
print(f"  Single split F1:       {single_f1*100:.2f}%  ← could be lucky or unlucky")
print(f"  Cross-validation F1:   {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%  ← reliable")
print()
print("In real projects: always report cross-validation scores, not single split.")


---
## ⚙️ Section 2: Hyperparameter Tuning

Every ML model has **hyperparameters** — settings you choose before training.

Examples for Random Forest:
- `n_estimators` — how many trees?
- `max_depth` — how deep each tree?
- `min_samples_split` — minimum samples to split a node?

So far you set these manually by guessing.
**GridSearchCV** automates this — it tries every combination and finds the best.

```
n_estimators: [50, 100, 200]
max_depth:    [5, 10, None]

GridSearch tries all 9 combinations:
(50, 5), (50, 10), (50, None),
(100, 5), (100, 10), (100, None),
(200, 5), (200, 10), (200, None)

Each evaluated with 5-fold CV → 45 total model runs
Best combination → returned automatically
```

**NestJS analogy:** like running your integration tests across all combinations
of config settings and picking the best performing config automatically.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [5, 10, None],
    "min_samples_split": [2, 5],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,              # 5-fold cross validation for each combination
    scoring='f1',
    n_jobs=-1,         # run in parallel
    verbose=1          # print progress
)

grid_search.fit(X_train, y_train)

print("\n=== GRID SEARCH RESULTS ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1:      {grid_search.best_score_*100:.2f}%")


In [ ]:
# ✅ Use the best model to predict on test data

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
best_test_f1 = f1_score(y_test, y_pred_best)

print(f"Best model test F1:    {best_test_f1*100:.2f}%")
print(f"Default model test F1: {single_f1*100:.2f}%")
print(f"Improvement:           +{(best_test_f1 - single_f1)*100:.2f}%")


In [ ]:
# ✅ See all results — not just the best

results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[[
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_split",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

print("Top 5 parameter combinations:")
print(results_df.head(5).to_string(index=False))


---
## 🔧 Section 3: Sklearn Pipeline — Production-Ready Code

So far your ML workflow looks like this:

```python
# Step 1 — scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Step 2 — train
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Step 3 — predict
y_pred = model.predict(X_test_scaled)
```

This is error-prone. Easy to forget to scale test data (you saw this bug already!).

**Pipeline** chains steps together into one object:
```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression())
])
pipeline.fit(X_train, y_train)      # automatically scales then trains
pipeline.predict(X_test)             # automatically scales then predicts
```

No manual scaling. No forgetting steps. No bugs.
This is how ML code looks in production.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ✅ Build a pipeline: Scale → Logistic Regression
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(random_state=42, max_iter=1000))
])

# fit() automatically: fits scaler on X_train, then fits model on scaled X_train
lr_pipeline.fit(X_train, y_train)

# predict() automatically: scales X_test, then predicts
lr_pred = lr_pipeline.predict(X_test)

print("=== Logistic Regression Pipeline ===")
print(f"F1 Score: {f1_score(y_test, lr_pred)*100:.2f}%")
print()
print(classification_report(y_test, lr_pred, target_names=["Not Viral", "Viral"]))


In [ ]:
# ✅ Random Forest pipeline (no scaler needed, but Pipeline still keeps things clean)

rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

print(f"Random Forest Pipeline F1: {f1_score(y_test, rf_pred)*100:.2f}%")


In [ ]:
# ✅ POWER MOVE: GridSearchCV + Pipeline together
# This is the real production pattern

# Note: when using a pipeline, prefix param names with the step name + '__'
# 'model__n_estimators' = the n_estimators param of the 'model' step

pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid_pipeline = {
    'model__n_estimators': [50, 100],
    'model__max_depth':    [5, 10, None],
}

grid_pipeline = GridSearchCV(
    pipeline,
    param_grid_pipeline,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_pipeline.fit(X_train, y_train)

print(f"\nBest params:  {grid_pipeline.best_params_}")
print(f"Best CV F1:   {grid_pipeline.best_score_*100:.2f}%")
print(f"Test F1:      {f1_score(y_test, grid_pipeline.predict(X_test))*100:.2f}%")


---
## 🏁 Section 4: Full Production ML Workflow — All Together

In [ ]:
# ✅ This is what a clean, production ML script looks like
# Every step is explicit, reproducible, and safe

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import numpy as np

# --- 1. Data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 2. Define candidate models as pipelines ---
candidates = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(random_state=42, max_iter=1000))
    ]),
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ))
    ]),
}

# --- 3. Evaluate each with cross-validation ---
print("=" * 55)
print("   MODEL COMPARISON — 5-Fold Cross Validation")
print("=" * 55)

best_name, best_score, best_pipeline = None, 0, None

for name, pipeline in candidates.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    mean, std = scores.mean(), scores.std()
    print(f"  {name:<25} CV F1: {mean*100:.2f}% ± {std*100:.2f}%")
    if mean > best_score:
        best_score = mean
        best_name  = name
        best_pipeline = pipeline

# --- 4. Train best model on full training data ---
print(f"\n🏆 Best model: {best_name}")
best_pipeline.fit(X_train, y_train)

# --- 5. Final evaluation on held-out test set ---
y_pred = best_pipeline.predict(X_test)
print(f"\nFinal Test F1: {f1_score(y_test, y_pred)*100:.2f}%")
print()
print(classification_report(y_test, y_pred, target_names=["Not Viral", "Viral"]))


In [ ]:
# ✅ Predict on new data with the final pipeline — one clean call

new_posts = pd.DataFrame({
    "likes":           [12000, 300,  5000],
    "comments":        [890,   15,   420],
    "shares":          [650,   8,    180],
    "hashtag_count":   [5,     25,   10],
    "caption_length":  [120,   8,    200],
    "hour_posted":     [18,    3,    12],
    "follower_count":  [50000, 800,  12000],
    "is_video":        [1,     0,    1],
    "engagement_rate": [0.256, 0.41, 0.188],
    "shares_per_like": [0.054, 0.026, 0.036],
})

predictions   = best_pipeline.predict(new_posts)
probabilities = best_pipeline.predict_proba(new_posts)

print("=== FINAL VIRAL PREDICTOR ===")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    label = "🔥 VIRAL" if pred == 1 else "❌ NOT VIRAL"
    print(f"Post {i+1}: {label}  ({prob[1]*100:.1f}% confidence)")


---
## 🎯 Section 5: Exercises

### Exercise 1
Run cross-validation on a single Decision Tree (`max_depth=5`) and compare
its mean CV F1 and std dev to Random Forest.
Which is more stable (lower std dev)? Why does that make sense?

### Exercise 2
Build a pipeline with `StandardScaler` + `LogisticRegression`
and run `GridSearchCV` on it to tune `model__C` (try `[0.01, 0.1, 1, 10]`).
`C` controls regularization — smaller C = simpler model.
What value of C works best?

### Exercise 3
Write a function `evaluate_model(pipeline, X, y)` that:
- Runs 5-fold cross-validation
- Prints mean F1, std dev, and a simple pass/fail
- Returns the mean F1 score
```python
# Expected output:
# Mean F1: 91.23% ± 1.45%
# Status: ✅ PASS (above 80% threshold)
```
> 💡 Exercise 3 is how you'd write a reusable evaluation utility
> in a real ML codebase — exactly like a utility function in NestJS.

In [ ]:
# 🏋️ YOUR WORKSPACE

# ✏️ Exercise 1:


# ✏️ Exercise 2:


# ✏️ Exercise 3:



---
## ✅ Final Week Checklist

- [ ] Can explain why cross-validation is more reliable than a single split
- [ ] Can run `cross_val_score` and interpret mean + std dev
- [ ] Know what hyperparameters are and how GridSearchCV finds the best ones
- [ ] Can build a sklearn Pipeline with multiple steps
- [ ] Understand why Pipeline prevents the scaler bug from Week 3-4
- [ ] Can combine Pipeline + GridSearchCV + CrossVal into one workflow
- [ ] Completed all 3 exercises

---

## 🎉 PHASE 2 COMPLETE!

Here's everything you now know:

| Week | What you learned |
|---|---|
| 1–2 | Linear Regression — predict numbers |
| 3–4 | Logistic Regression — classify yes/no |
| 5–6 | Decision Trees + Random Forest — powerful ensemble model |
| Final | Cross-val + Tuning + Pipelines — production ML workflow |

---

## 🔜 Phase 3: Deep Learning

This is where things get genuinely exciting:
- **Neural Networks** — how they work, how to build one
- **PyTorch basics** — the industry standard DL framework
- **CNNs** — image models
- **NLP + Transformers** — text models (this is where your Bangla spam dataset comes back!)
- **HuggingFace** — use pre-trained models in 10 lines of code
- **LLM APIs** — build apps on top of GPT/Claude/Gemini

> 💬 Finish the exercises and come back — Phase 3 is the most exciting part!